In [22]:
import pandas as pd
import numpy as np

import sys
sys.path.append('../')

from sqlalchemy import create_engine, text

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

ENGINE = create_engine(f'postgresql+psycopg://{USER_DB}:{PASS_DB}@{HOST_DB}:{PORT_DB}/{NAME_DB}')

In [23]:
#path_xlsx_avance = r'G:/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - AVANCE COSECHA/2024/AVANCE_SEMANAL/AVANCE DE COSECHA V1.xlsx'
path_xlsx_avance = r'G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\AVANCE SEMANAL\AVANCE DE COSECHA_V1.xlsx'
xlsx_resumen = pd.read_excel(path_xlsx_avance, sheet_name='AVANCE_FECHAS')
xlsx_resumen

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
0,1,2026-06-05 00:00:00,1.560303,0.00,NaN
1,1,(en blanco),583.665310,0.00,NaN
2,2,2026-06-30 00:00:00,10.134687,608.74,735.19
3,2,2026-07-05 00:00:00,16.464981,1092.40,950.47
4,2,(en blanco),174.199061,0.00,NaN
...,...,...,...,...,...
1582,(en blanco),2026-06-20 00:00:00,5.668316,227.26,286.68
1583,(en blanco),2026-06-30 00:00:00,4.184842,221.77,742.23
1584,(en blanco),2026-07-05 00:00:00,NaN,NaN,183.08
1585,(en blanco),(en blanco),86.696082,0.00,NaN


In [28]:
xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')
# Convertir a número (los no válidos se vuelven NaN)
xlsx_resumen['COD_COS'] = pd.to_numeric(xlsx_resumen['COD_COS'], errors='coerce')
# Eliminar filas con NaN
xlsx_resumen = xlsx_resumen.dropna(subset=['COD_COS'])
# Convertir a int
xlsx_resumen['COD_COS'] = xlsx_resumen['COD_COS'].astype(int)
xlsx_resumen

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGADO
0,1,2026-06-05 00:00:00,1.560303,0.00,NaN
1,1,(en blanco),583.665310,0.00,NaN
2,2,2026-06-30 00:00:00,10.134687,608.74,735.19
3,2,2026-07-05 00:00:00,16.464981,1092.40,950.47
4,2,(en blanco),174.199061,0.00,NaN
...,...,...,...,...,...
1577,668,2026-06-30 00:00:00,15.764304,1083.08,2334.35
1578,668,2026-07-05 00:00:00,NaN,NaN,1071.55
1579,668,(en blanco),44.604462,0.00,NaN
1580,669,2026-06-30 00:00:00,5.598136,394.94,NaN


In [29]:
# 1. Intentar convertir a fecha (los no válidos quedan como NaT)
xlsx_resumen['FECHA'] = pd.to_datetime(xlsx_resumen['FECHA'], errors='coerce')
# Eliminar filas con fechas inválidas
xlsx_resumen = xlsx_resumen.dropna(subset=['FECHA'])

In [30]:
def colapsar_tabla(df_param):
    suma_acumulador = [None] * len(df_param)
    acumulador = 0
    contador = 0
    for i, r in df_param.iterrows():
        if(r['AREA'] == 0):
            acumulador += r['ENTREGADO']
            contador += 1
        else:
            acumulador += r['ENTREGADO']
            suma_acumulador[contador] = acumulador
            acumulador = 0
            contador += 1
            continue
    if suma_acumulador[-1] == None:
        suma_acumulador[-1] = acumulador

    df_param['ENTREGAS'] = suma_acumulador
    
    df_param = df_param.dropna(subset=['ENTREGAS'])
    df_param = df_param.drop(columns=['ENTREGADO'])
    #df_param = df_param[df_param['AREA']!=0]
    return df_param

In [31]:
list_cod_cos = list(set(xlsx_resumen['COD_COS']))

In [32]:
list_df = []

In [33]:
for i in list_cod_cos:
    grupo = xlsx_resumen[xlsx_resumen['COD_COS']==i].fillna(0)
    result = colapsar_tabla(grupo)
    list_df.append(result)

In [34]:
df_combinado = pd.concat(list_df, ignore_index=True)

In [35]:
df_combinado.to_excel(r'G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\AVANCE SEMANAL\_TEMP_AVANCE_FECHAS.xlsx', index=False, engine='openpyxl', sheet_name='data')
#df_combinado.to_excel('G:/Ingenio Azucarero Guabira S.A/UTEA - SEMANAL - AVANCE COSECHA/2024/AVANCE_SEMANAL/_TEMP_AVANCE_FECHAS.xlsx', index=False, engine='openpyxl', sheet_name='data')

In [36]:
df_combinado.head(2)

,COD_COS,FECHA,AREA,ESTIMADO,ENTREGAS
0,1,2026-06-05,1.560303,0.00,0.00
1,2,2026-06-30,10.134687,608.74,735.19


In [38]:
def cargar_avance_fechas_a_db(df):
    df = df.rename(columns={
            'COD_COS': 'cod_cos',
            'FECHA': 'fecha',
            'AREA': 'area',
            'ESTIMADO': 'estimado',
            'ENTREGAS': 'entregas'
    })
    
    df['cod_cos']     = df['cod_cos'].astype('Int64')
    #df['fecha']   = pd.to_datetime(df["fecha"])
    df['area']  = df['area'].astype('float')
    df['estimado']   = df['estimado'].astype('float')
    df['entregas']   = df['entregas'].astype('float')

    #validad divicion entre 0
    df["tch_estimado"] = np.where(df["area"] > 0,
                              df["estimado"] / df["area"],
                              0)
    #validad divicion entre 0
    df["tch_entregas"] = np.where(df["area"] > 0,
                              df["entregas"] / df["area"],
                              0)
    
    df['diferencia_tn'] = df['entregas'] - df['estimado']
    df["porcen_dif"] = np.where(df["entregas"] > 0,
                              df["diferencia_tn"] / df["entregas"],
                              0)

    with ENGINE.begin() as conn:  # Inicia transacción
        # Vaciar la tabla y reiniciar secuencia
        conn.execute(text(f'TRUNCATE TABLE catastro_iag.data_avance_fechas RESTART IDENTITY'))
        
        # Insertar nuevos datos
        df.to_sql(
            name='data_avance_fechas',
            con=conn,  # conexión cruda dentro de la transacción
            schema='catastro_iag',
            if_exists='append',
            index=False,
            method='multi',
            chunksize=1000
        )
    print(f"✅ Se ha actualziado la tabla de codigos cosecha")

In [39]:
cargar_avance_fechas_a_db(df_combinado)

✅ Se ha actualziado la tabla de codigos cosecha
